In [ ]:
import os
import gc
from pathlib import Path
from typing import List
from dataclasses import dataclass
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import sys
sys.path.append('/kaggle/input/hull-tactical-market-prediction')
import kaggle_evaluation.default_inference_server

# ============ 1. 全局配置与类定义 ============

# 在 Kaggle 环境中通常不需要修改这里，但在本地可以指向 './data'
if os.path.exists('/kaggle/input/hull-tactical-market-prediction/'):
    DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')
else:
    DATA_PATH = Path('./data')

MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0
SIGNAL_MULTIPLIER = 400.0

LGBM_PARAMS = {
    "n_estimators": 5000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "num_leaves": 128,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "regression",
    "metric": "rmse",
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
    "min_child_samples": 10
}

@dataclass
class DatasetOutput:
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    features: List[str]
    feature_means: dict
    scaler: StandardScaler

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float
    min_signal: float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL

VARS_TO_KEEP = [
    "M4", "P6", "M3", "P3", "P4", "E19", "P13", "I2", "S8", "P7",
    "V13", "S12", "S6", "E12", "E4"
]
# ============ 2. 数据处理与特征工程函数 ============

def load_trainset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(pl.exclude("date_id").cast(pl.Float64, strict=False))
        .sort("date_id")
    )

def load_testset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(pl.exclude("date_id").cast(pl.Float64, strict=False))
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """训练时特征工程：使用 ewm_mean 填充"""
    vars_to_keep = VARS_TO_KEEP
    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5)) for col in vars_to_keep
        ])
        .drop_nulls()
    )

def create_example_dataset_inference(df: pl.DataFrame, feature_means: dict) -> pl.DataFrame:
    """推理时特征工程：使用训练均值填充（更稳健）"""
    vars_to_keep = VARS_TO_KEEP
    
    # 填充基础列以计算衍生特征
    base_cols = ["I1", "I2", "I7", "I9", "M11"]
    fill_base_exprs = []
    for col in base_cols:
        mean_val = feature_means.get(col, 0.0)
        if col in df.columns:
            fill_base_exprs.append(pl.col(col).fill_null(mean_val))
        else:
            fill_base_exprs.append(pl.lit(mean_val).alias(col))
    
    if fill_base_exprs:
        df = df.with_columns(fill_base_exprs)

    # 计算衍生特征 U1, U2
    df = df.with_columns(
        (pl.col("I2") - pl.col("I1")).alias("U1"),
        (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3 + 1e-8)).alias("U2")
    )
    
    # 选择列并处理最终特征的缺失值
    df = df.select(["date_id", "target"] + vars_to_keep) if "target" in df.columns else df.select(["date_id"] + vars_to_keep)
    
    fill_exprs = []
    for col in vars_to_keep:
        mean_val = feature_means.get(col, 0.0)
        # 对可能产生的 Inf/NaN 进行处理
        fill_exprs.append(
            pl.when(pl.col(col).is_null() | pl.col(col).is_infinite())
            .then(mean_val)
            .otherwise(pl.col(col))
            .alias(col)
        )
    return df.with_columns(fill_exprs)

def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    common_columns = [col for col in train.columns if col in test.columns]
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_and_process_dataset(train: pl.DataFrame, test: pl.DataFrame) -> DatasetOutput:
    df = join_train_test_dataframes(train, test)
    df = create_example_dataset(df=df)
    
    train_ids = train.get_column('date_id').to_list()
    test_ids = test.get_column('date_id').to_list()
    
    train_processed = df.filter(pl.col('date_id').is_in(train_ids))
    test_processed = df.filter(pl.col('date_id').is_in(test_ids))
    
    features = [col for col in test_processed.columns if col not in ['date_id', 'target']]
    
    # 计算均值供推理使用
    all_cols_for_means = features + ["I1", "I2", "I7", "I9", "M11"]
    feature_means = {}
    for col in all_cols_for_means:
        if col in train_processed.columns:
            val = train_processed.select(pl.col(col).mean()).item()
            feature_means[col] = val if val is not None else 0.0
        else:
            feature_means[col] = 0.0
            
    X_train = train_processed.select(features).to_pandas()
    y_train = train_processed.select("target").to_pandas()["target"]
    X_test = test_processed.select(features).to_pandas()
    y_test = test_processed.select("target").to_pandas()["target"]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return DatasetOutput(
        X_train=pd.DataFrame(X_train_scaled, columns=features, index=X_train.index),
        X_test=pd.DataFrame(X_test_scaled, columns=features, index=X_test.index),
        y_train=y_train, y_test=y_test, features=features, feature_means=feature_means, scaler=scaler
    )

# ============ 3. 模型训练逻辑 ============

def train_model_pipeline():
    print("Loading and processing data...")
    train_df = load_trainset()
    test_df = load_testset()
    dataset = split_and_process_dataset(train_df, test_df)
    
    print("Training LGBM model...")
    # 这里为了演示直接使用全量数据训练（省略了CV步骤以缩短提交代码长度，可按需加回）
    train_data = lgb.Dataset(dataset.X_train, label=dataset.y_train)
    model = lgb.train(LGBM_PARAMS, train_data)
    
    return model, dataset

# ============ 4. 初始化（在服务器启动前运行） ============

# 执行训练流程，获取模型和数据集配置
# 注意：这部分代码会在提交环境启动时运行一次
try:
    model, dataset = train_model_pipeline()
    ret_signal_params = RetToSignalParameters(signal_multiplier=SIGNAL_MULTIPLIER)
    print("Model ready.")
except Exception as e:
    print(f"Error during model initialization: {e}")
    # 这里可以添加 fallback 逻辑，例如加载预训练模型

# ============ 5. 推理函数 (API 接口) ============

def predict(test: pl.DataFrame) -> float:
    """
    Kaggle 评估服务器调用的推理函数
    """
    # 1. 预处理
    # 这里的 test 是一个 batch (通常是一行或多行)，需要转为 float
    test = test.with_columns(pl.exclude("date_id", "row_id").cast(pl.Float64, strict=False))
    if "lagged_forward_returns" in test.columns:
        test = test.rename({'lagged_forward_returns':'target'})
    
    # 2. 特征工程 (使用全局 dataset 中的 means)
    df = create_example_dataset_inference(test, dataset.feature_means)
    
    # 3. 准备 Pandas DataFrame
    try:
        # 确保列顺序与训练时一致
        X_test = df.select(dataset.features).to_pandas()
    except Exception:
        # 容错：补充缺失列
        current_cols = df.columns
        for c in dataset.features:
            if c not in current_cols:
                df = df.with_columns(pl.lit(0.0).alias(c))
        X_test = df.select(dataset.features).to_pandas()
        
    # 4. 标准化
    X_test_scaled = dataset.scaler.transform(X_test)
    
    # 5. 预测
    raw_pred = model.predict(X_test_scaled)[0]
    
    # 6. 信号转换
    signal = np.clip(
        raw_pred * ret_signal_params.signal_multiplier + 1, 
        ret_signal_params.min_signal, 
        ret_signal_params.max_signal
    )
    
    return float(signal)

# ============ 6. 启动服务器 ============

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    # 本地测试网关
    inference_server.run_local_gateway(
        (str(DATA_PATH),)
    )
Loading and processing data...
Training LGBM model...
Model ready.

Loading and processing data...
Training LGBM model...
Model ready.
